In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/dataup1/ogbn-arxiv/RELEASE_v1.txt
/kaggle/input/datasets/dataup1/ogbn-arxiv/ogbn-master.csv
/kaggle/input/datasets/dataup1/ogbn-arxiv/split/time/valid.csv
/kaggle/input/datasets/dataup1/ogbn-arxiv/split/time/train.csv
/kaggle/input/datasets/dataup1/ogbn-arxiv/split/time/test.csv
/kaggle/input/datasets/dataup1/ogbn-arxiv/mapping/labelidx2arxivcategeory.csv
/kaggle/input/datasets/dataup1/ogbn-arxiv/mapping/README.md
/kaggle/input/datasets/dataup1/ogbn-arxiv/mapping/nodeidx2paperid.csv
/kaggle/input/datasets/dataup1/ogbn-arxiv/processed/pre_filter.pt
/kaggle/input/datasets/dataup1/ogbn-arxiv/processed/pre_transform.pt
/kaggle/input/datasets/dataup1/ogbn-arxiv/processed/geometric_data_processed.pt
/kaggle/input/datasets/dataup1/ogbn-arxiv/raw/num-edge-list.csv
/kaggle/input/datasets/dataup1/ogbn-arxiv/raw/num-node-list.csv
/kaggle/input/datasets/dataup1/ogbn-arxiv/raw/node-feat.csv
/kaggle/input/datasets/dataup1/ogbn-arxiv/raw/edge.csv
/kaggle/input/datasets/dataup1/

In [2]:
import torch

# 1. Identify your specific environment
torch_ver = torch.__version__.split('+')[0]
cuda_ver = torch.version.cuda.replace('.', '')

print(f"Installing for Torch {torch_ver} and CUDA {cuda_ver}...")

# 2. Install the four "hard" dependencies from the official PyG wheel index
!pip install --quiet torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-{torch_ver}+cu{cuda_ver}.html

# 3. Install/Reinstall the main libraries
!pip install --quiet torch-geometric ogb

Installing for Torch 2.10.0 and CUDA 128...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 92.1 MB/s eta 0:00:0000:010:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 85.6 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 90.2 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 54.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 1.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 14.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 6.5 MB/s eta 0:00:00


In [3]:
import torch
import os.path as osp
from ogb.nodeproppred import PygNodePropPredDataset
import torch_geometric.transforms as T

# A more robust monkeypatch
_orig_load = torch.load

def smart_load(*args, **kwargs):
    # Ensure weights_only is False, even if it's already in kwargs
    kwargs['weights_only'] = False
    return _orig_load(*args, **kwargs)

torch.load = smart_load

try:
    # 1. Define transforms
    # ToUndirected makes the graph symmetric, which helps GCNs learn better
    transform = T.Compose([T.ToSparseTensor(), T.ToUndirected()])
    
    # 2. Load the dataset
    dataset = PygNodePropPredDataset(name='ogbn-arxiv', transform=transform)
    data = dataset[0]
    
    # 3. Get the splits provided by OGB
    split_idx = dataset.get_idx_split()
    train_idx = split_idx['train']
    valid_idx = split_idx['valid']
    test_idx = split_idx['test']
    
    print("✅ Success! Dataset loaded and splits ready.")
    print(f"Graph stats: {data.num_nodes} nodes, {data.num_edges} edges")
    print(f"Training nodes: {train_idx.numel()}")
finally:
    # Restore the original function to keep the rest of your notebook 'clean'
    torch.load = _orig_load

Downloaded 0.08 GB: 100%|██████████| 81/81 [00:03<00:00, 20.44it/s]


Extracting dataset/arxiv.zip


Processing...


Loading necessary files...
This might take a while.
Processing graphs...


100%|██████████| 1/1 [00:00<00:00, 11214.72it/s]


Converting graphs into PyG objects...


100%|██████████| 1/1 [00:00<00:00, 424.78it/s]

Saving...



Done!


✅ Success! Dataset loaded and splits ready.
Graph stats: 169343 nodes, 1166243 edges
Training nodes: 90941


In [24]:
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch.nn import BatchNorm1d

class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers):
        super(GCN, self).__init__()
        self.convs = torch.nn.ModuleList()
        self.bns = torch.nn.ModuleList()
        
        self.convs.append(GCNConv(in_channels, hidden_channels))
        self.bns.append(BatchNorm1d(hidden_channels))
        
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(hidden_channels, hidden_channels))
            self.bns.append(BatchNorm1d(hidden_channels))
            
        self.convs.append(GCNConv(hidden_channels, out_channels))

    def forward(self, x, adj_t):
        for i, conv in enumerate(self.convs[:-1]):
            h = conv(x, adj_t)
            h = self.bns[i](h)
            h = F.relu(h)
            h = F.dropout(h, p=0.5, training=self.training)
            # Residual connection: adds original x back to the new features h
            x = x + h if x.shape == h.shape else h 
            
        x = self.convs[-1](x, adj_t)
        return x.log_softmax(dim=-1)

# Initialize with weight_decay to prevent overfitting
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-5)

# Initialize Model & Optimizer
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = GCN(dataset.num_features, 256, dataset.num_classes, 3).to(device)
data = data.to(device)

In [25]:
from ogb.nodeproppred import Evaluator

evaluator = Evaluator(name='ogbn-arxiv')

In [26]:
def train():
    model.train()
    optimizer.zero_grad()
    
    # Forward pass: the model looks at the whole graph to pass messages
    out = model(data.x, data.adj_t)
    
    # Loss calculation: only look at training indices
    # data.y is [N, 1], so we squeeze it to [N] for the loss function
    loss = F.nll_loss(out[train_idx], data.y[train_idx].squeeze(1))
    
    loss.backward()
    optimizer.step()
    
    return loss.item()

In [27]:
@torch.no_grad()
def test():
    model.eval()
    
    # Get predictions for all nodes
    out = model(data.x, data.adj_t)
    y_pred = out.argmax(dim=-1, keepdim=True)

    # Use the evaluator to get accuracy for each split
    train_acc = evaluator.eval({
        'y_true': data.y[train_idx],
        'y_pred': y_pred[train_idx],
    })['acc']
    
    valid_acc = evaluator.eval({
        'y_true': data.y[val_idx],
        'y_pred': y_pred[val_idx],
    })['acc']
    
    test_acc = evaluator.eval({
        'y_true': data.y[test_idx],
        'y_pred': y_pred[test_idx],
    })['acc']

    return train_acc, valid_acc, test_acc

In [28]:
# Create an alias so both names work
val_idx = valid_idx

# Now your training loop will find 'val_idx' without issues!

In [29]:
for epoch in range(1, 101):
    loss = train()
    
    # Evaluate every 10 epochs to see progress
    if epoch % 10 == 0:
        result = test()
        train_acc, val_acc, test_acc = result
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, '
              f'Train: {train_acc:.4f}, Val: {val_acc:.4f}, Test: {test_acc:.4f}')

Epoch: 010, Loss: 4.9714, Train: 0.0111, Val: 0.0107, Test: 0.0119
Epoch: 020, Loss: 4.9862, Train: 0.0137, Val: 0.0139, Test: 0.0176
Epoch: 030, Loss: 4.9684, Train: 0.0204, Val: 0.0172, Test: 0.0237
Epoch: 040, Loss: 4.9711, Train: 0.0291, Val: 0.0206, Test: 0.0246
Epoch: 050, Loss: 4.9854, Train: 0.0358, Val: 0.0219, Test: 0.0243
Epoch: 060, Loss: 4.9748, Train: 0.0396, Val: 0.0223, Test: 0.0240
Epoch: 070, Loss: 4.9835, Train: 0.0409, Val: 0.0224, Test: 0.0238
Epoch: 080, Loss: 4.9817, Train: 0.0415, Val: 0.0226, Test: 0.0237
Epoch: 090, Loss: 4.9901, Train: 0.0417, Val: 0.0228, Test: 0.0237
Epoch: 100, Loss: 4.9806, Train: 0.0418, Val: 0.0229, Test: 0.0238
